# 타이타닉 생존자 예측 - 주석이 추가된 학습용 노트북
원본 `2_2.ipynb`에 각 셀의 동작을 설명하는 주석을 추가했습니다. 원본 코드에 존재하는 버그/주의사항도 함께 표시했습니다.

## 1. 라이브러리 임포트

In [ ]:
import numpy as np   # 수치 연산 및 배열 처리를 위한 라이브러리
import pandas as pd  # 데이터프레임 기반 데이터 처리를 위한 라이브러리
import matplotlib.pyplot as plt  # 그래프 시각화를 위한 라이브러리
import seaborn as sns  # matplotlib 기반의 통계 시각화 라이브러리

## 2. 데이터 로드

In [ ]:
# train.csv 파일을 읽어 데이터프레임으로 저장
titanic_df = pd.read_csv('./train.csv')
titanic_df.head()  # 상위 5개 행을 출력해서 데이터 구조를 확인

## 3. 데이터 구조 및 결측치 확인

In [ ]:
# 각 컬럼의 데이터 타입(dtype)과 결측치가 아닌 값의 개수(Non-Null Count)를 확인
# Age(714/891), Cabin(204/891), Embarked(889/891)에 결측치가 있음을 알 수 있음
titanic_df.info()

## 4. 결측치 처리

In [ ]:
# Age 컬럼의 결측치를 전체 평균 나이로 채움
titanic_df['Age'] = titanic_df['Age'].fillna(titanic_df['Age'].mean())
# Cabin 컬럼의 결측치를 'N'(선실 정보 없음을 의미하는 문자)으로 채움
titanic_df['Cabin'] = titanic_df['Cabin'].fillna('N')
# Embarked 컬럼의 결측치를 'N'으로 채움
titanic_df['Embarked'] = titanic_df['Embarked'].fillna('N')

titanic_df.isnull().sum() # 결측치 확인 (모든 컬럼이 0이 되어야 정상)

## 5. 간단한 데이터 탐색(EDA)

In [ ]:
# Sex 컬럼의 값별 개수 확인 (남성/여성 분포)
titanic_df['Sex'].value_counts()

In [ ]:
# Cabin 컬럼의 값별 개수 확인
# 결측치를 'N'으로 채웠기 때문에 'N'이 687건으로 압도적으로 많고,
# 나머지는 선실 번호(예: 'C23 C25 C27')가 거의 전부 달라 종류가 매우 많음
titanic_df['Cabin'].value_counts()

In [ ]:
# Embarked(탑승 항구) 컬럼의 값별 개수 확인
# S=Southampton, C=Cherbourg, Q=Queenstown, N=결측치였던 값
titanic_df['Embarked'].value_counts()

## 6. Cabin 값 단순화
객실 번호 전체 대신 맨 앞 알파벳(선실 구역)만 남겨서 범주 종류를 줄입니다.

In [ ]:
# Cabin 값에서 첫 글자만 추출하여 선실 구역으로 단순화
# 예: 'C85' -> 'C', 'N' -> 'N'
titanic_df['Cabin'] = titanic_df['Cabin'].str[:1]

titanic_df['Cabin'].head()

## 7. 성별 x 생존여부 교차 집계

In [ ]:
# 성별(Sex)과 생존여부(Survived)로 그룹화하여 각 그룹의 인원수를 집계
# 여성의 생존 비율이 남성보다 훨씬 높다는 것을 수치로 확인할 수 있음
titanic_df.groupby(['Sex', 'Survived'])['Survived'].count()

## 8. 객실 등급 x 성별에 따른 생존율 시각화

In [ ]:
# 객실 등급(Pclass)별, 성별(Sex)별 생존율을 막대그래프로 시각화
# 1등급 여성 승객의 생존율이 가장 높고, 3등급 남성 승객의 생존율이 가장 낮은 경향을 보임
sns.barplot(x='Pclass', y= 'Survived', hue='Sex', data=titanic_df)

## 9. 연령대 구간화 및 생존율 시각화
나이를 구간(범주)으로 나누는 함수를 만들고, 연령대별 생존율을 시각화합니다.

**주의(버그 가능성)**
- `get_category` 함수는 나이가 60을 초과하는 경우를 처리하는 분기가 없어 빈 문자열(`""`)을 반환합니다.
- `group_name` 리스트의 `'unknow'`, `'Tennager'`는 함수가 실제로 반환하는 `'unknown'`, `'Teenager'`와 철자가 달라, `order` 옵션으로 해당 범주의 막대가 그래프에 정상적으로 표시되지 않을 수 있습니다.

In [ ]:
# 나이(age)를 입력받아 해당하는 연령대 범주(문자열)를 반환하는 함수
def get_category(age) :
    cat = ""
    if age <= -1 : cat = 'unknown'          # 음수 나이: 알 수 없음(실제로는 발생하지 않음)
    elif age <= 5 : cat = 'baby'            # 5세 이하: 아기
    elif age <= 12 : cat = 'Child'          # 12세 이하: 어린이
    elif age <= 18 : cat = 'Teenager'       # 18세 이하: 청소년
    elif age <= 25 : cat = 'student'        # 25세 이하: 대학생 나이대
    elif age <= 35 : cat = 'Young Adult'    # 35세 이하: 청년
    elif age <= 60 : cat = 'Adult'          # 60세 이하: 성인
    # 60세 초과 분기가 없어 이 경우 cat은 빈 문자열("")로 반환됨 (버그)

    return cat

# 그래프의 x축에 표시할 범주 순서를 지정하는 리스트
# 'unknow', 'Tennager'는 오타로, 함수가 반환하는 'unknown', 'Teenager'와 일치하지 않음
group_name = ['unknow', 'baby', 'Child', 'Tennager', 'student', 'Young Adult', 'Adult']

# Age 값에 get_category를 적용하여 Age_cat(연령대) 파생 컬럼 생성
titanic_df['Age_cat'] = titanic_df['Age'].apply(lambda x : get_category(x))

plt.figure(figsize=(10, 6))
# 연령대(Age_cat)별, 성별(Sex)별 생존율 시각화
sns.barplot(x='Age_cat', y = 'Survived', hue='Sex', data=titanic_df, order = group_name)

# 시각화에만 사용한 임시 컬럼 Age_cat을 다시 삭제 (원본 데이터프레임 보존)
titanic_df.drop('Age_cat', axis= 1, inplace = True)

## 10. 범주형 데이터 인코딩
머신러닝 모델은 문자열을 직접 입력받을 수 없으므로, 문자열 컬럼을 숫자로 변환(레이블 인코딩)합니다.

In [ ]:
from sklearn import preprocessing  # 레이블 인코딩 등 전처리 도구가 들어있는 모듈

# 문자열(범주형) 컬럼들을 숫자로 변환(레이블 인코딩)하는 함수
def encode_features(df) :
    features = ['Cabin', 'Sex', 'Embarked']  # 인코딩할 범주형 컬럼 목록
    for feature in features:
        le = preprocessing.LabelEncoder()
        le = le.fit(df[feature])                 # 해당 컬럼의 고유값들을 학습(각 값에 숫자를 매핑)
        df[feature]= le.transform(df[feature])    # 학습된 매핑 규칙으로 값을 숫자로 변환
    return df

titanic_df = encode_features(titanic_df)
titanic_df.head()

In [ ]:
# 인코딩 후 Cabin 값 분포를 다시 확인 (문자 'N','C',... 대신 숫자로 바뀜)
titanic_df['Cabin'].value_counts()

In [ ]:
titanic_df.head()

## 11. 불필요한 컬럼 제거
모델 학습에 의미가 없는 식별자/텍스트 컬럼(PassengerId, Name, Ticket)을 제거합니다.

**주의**: 이 셀을 한 번 실행한 뒤 다시 실행하면 해당 컬럼이 이미 삭제된 상태이므로 `KeyError`가 발생합니다(원본 노트북 출력에서 확인됨). 노트북 셀은 다시 실행하기 전에 커널을 재시작하거나, `errors='ignore'` 옵션을 사용하는 것이 안전합니다.

In [ ]:
# 모델 학습에 불필요한 식별/텍스트 컬럼 제거
titanic_df.drop(['PassengerId', 'Name', 'Ticket'], axis= 1, inplace=True)
titanic_df.head()

In [ ]:
# Fare(요금) 컬럼에 결측치가 남아있는지 확인 (0이면 결측치 없음)
titanic_df['Fare'].isna().sum()

## 12. 특징(X)/레이블(y) 분리
**주의(버그)**: 아래 코드는 두 줄 모두 변수명이 `y_titanic_df`로 되어 있습니다. 두 번째 줄이 첫 번째 줄의 결과를 덮어써 버리기 때문에, 의도와 달리 생존여부(label)가 아니라 나머지 특징들(feature)이 `y_titanic_df`에 저장됩니다. 원래 의도는 아래와 같아야 합니다.
```python
y_titanic_df = titanic_df['Survived']           # 정답 레이블
X_titanic_df = titanic_df.drop('Survived', axis=1)  # 입력 특징
```

In [ ]:
# 정답 레이블(y)과 입력 특징(X)을 분리하려는 의도였으나,
# 두 줄 모두 y_titanic_df에 대입되어 두 번째 줄의 값(Survived를 제외한 나머지 컬럼들)만 남게 됨
y_titanic_df = titanic_df['Survived']
y_titanic_df = titanic_df.drop('Survived', axis = 1)

## 13. 학습/테스트 데이터 분리
**주의(버그)**: 바로 위 셀의 변수명 오류 때문에 `X_titanic_df`가 한 번도 정의되지 않아 아래 코드를 실행하면 `NameError: name 'X_titanic_df' is not defined`가 발생합니다. 정상 동작을 위해서는 11번 셀을 아래처럼 고쳐야 합니다.
```python
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)
```

In [ ]:
from sklearn.model_selection import train_test_split

# X_titanic_df, y_titanic_df를 8:2 비율로 학습용/테스트용 데이터로 분리
# random_state를 고정하여 실행할 때마다 동일한 분리 결과가 나오도록 함
X_train, X_test, Y_train, Y_test = train_test_split(X_titanic_df, y_titanic_df, test_size=0.2, random_state= 11)

## 14. 여러 분류 모델 학습 및 정확도 비교
결정트리, 랜덤포레스트, 로지스틱 회귀 세 가지 모델을 학습하고 테스트 데이터에 대한 정확도를 비교합니다.

**주의(버그)**: 랜덤포레스트와 로지스틱 회귀를 각각 `fit`으로 학습시키지만, 예측(`predict`)은 세 경우 모두 `dt_clf`(결정트리)로 수행하고 있습니다. 즉 `rf_pred`, `lr_pred` 모두 실제로는 결정트리의 예측 결과이며, 랜덤포레스트/로지스틱 회귀의 성능을 보여주지 못합니다. 올바르게 고치면 `rf_clf.predict(X_test)`, `lr_clf.predict(X_test)`가 되어야 합니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier     # 결정트리 분류기
from sklearn.ensemble import RandomForestClassifier # 랜덤포레스트 분류기(여러 결정트리의 앙상블)
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 분류기
from sklearn.metrics import accuracy_score            # 정확도 계산 함수

dt_clf = DecisionTreeClassifier(random_state=11)
rf_clf = RandomForestClassifier(random_state=11)
lr_clf = LogisticRegression(solver= 'liblinear')

# 결정트리 학습 및 예측, 정확도 출력
dt_clf.fit(X_train, Y_train)
dt_pred = dt_clf.predict(X_test)
print('결정트리정확도 :', accuracy_score(Y_test, dt_pred))

# 랜덤포레스트를 학습(fit)하지만, 예측은 dt_clf로 수행하고 있음 (버그: rf_clf.predict(X_test) 여야 함)
rf_clf.fit(X_train, Y_train)
rf_pred = dt_clf.predict(X_test)
print('랜덤포레스트정확도 :', accuracy_score(Y_test, rf_pred))

# 로지스틱 회귀를 학습(fit)하지만, 예측은 dt_clf로 수행하고 있음 (버그: lr_clf.predict(X_test) 여야 함)
lr_clf.fit(X_train, Y_train)
lr_pred = dt_clf.predict(X_test)
print('로지스틱회귀정확도 :', accuracy_score(Y_test, lr_pred))

## 15. K-Fold 교차검증
데이터를 5개로 나누어 번갈아 학습/검증 데이터로 사용하면서 모델의 평균 성능을 측정합니다. 단일 train/test 분리보다 더 안정적인 성능 추정치를 얻을 수 있습니다.

In [ ]:
from sklearn.model_selection import KFold

# K-Fold 교차검증을 수행하는 함수
def exec_kfold(clf, folds = 5):
    kfold = KFold(n_splits= folds)  # 전체 데이터를 folds개의 부분집합(fold)으로 분할
    scores = []

    # 매 반복마다 1개의 fold를 검증용으로, 나머지를 학습용으로 사용
    for iter_count, (train_index, test_index) in enumerate(kfold.split(X_titanic_df)):
        X_train, X_test = X_titanic_df.values[train_index], X_titanic_df.values[test_index]
        Y_train, Y_test = y_titanic_df.values[train_index], y_titanic_df.values[test_index]

        clf.fit(X_train, Y_train)
        predictions = clf.predict(X_test)
        accuracy = accuracy_score(Y_test, predictions)
        scores.append(accuracy)
        print('교차검증정확도 :', accuracy)
    mean_score = np.mean(scores)  # 5번의 fold 정확도를 평균내어 전체 성능을 추정
    print('평균 정확도 :', mean_score)

exec_kfold(dt_clf, 5)  # 결정트리에 대해 5-Fold 교차검증 실행

In [ ]:
exec_kfold(rf_clf, 5)  # 랜덤포레스트에 대해 5-Fold 교차검증 실행

In [ ]:
exec_kfold(lr_clf, 5)  # 로지스틱 회귀에 대해 5-Fold 교차검증 실행

## 16. GridSearchCV를 이용한 하이퍼파라미터 튜닝
결정트리의 주요 하이퍼파라미터(`max_depth`, `min_samples_split`, `min_samples_leaf`) 조합을 교차검증(cv=5)으로 모두 탐색하여 가장 성능이 좋은 조합을 찾습니다.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 탐색할 하이퍼파라미터 값들의 범위를 딕셔너리로 정의
parameters = {'max_depth': [2,  3, 5, 10],
              'min_samples_split' : [2, 3, 5],
              'min_samples_leaf': [1, 5, 8]}

# 지정한 파라미터들의 모든 조합에 대해 5-Fold 교차검증을 수행하여 최적 조합을 탐색
grid_dclf = GridSearchCV(dt_clf, param_grid=parameters,
                         scoring='accuracy', cv = 5)
grid_dclf.fit(X_train, Y_train)

print(grid_dclf.best_params_)   # 교차검증 결과 가장 성능이 좋았던 하이퍼파라미터 조합
print(grid_dclf.best_score_)    # 그 조합의 교차검증 평균 정확도

best_dclf = grid_dclf.best_estimator_  # 최적 파라미터로 이미 학습이 완료된 모델

dpredicitions = best_dclf.predict(X_test)
accuracy = accuracy_score(Y_test, dpredicitions)

print(accuracy)  # 최적 모델의 홀드아웃 테스트 데이터에 대한 최종 정확도